# Adult Income kNN Classifier

This notebook builds a `scikit-learn` kNN baseline for predicting whether income is `>50K`.

Workflow:
- load the prepared CSV interface from notebook 02
- freeze a stratified internal holdout split before tuning
- tune kNN hyperparameters with cross-validation on the development split only
- evaluate the selected model once on the untouched holdout set
- refit on the full prepared training set and generate a competition-style submission


In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 160)


In [ ]:
def resolve_project_root() -> Path:
    candidates = [Path.cwd(), Path.cwd().parent]
    for candidate in candidates:
        if (candidate / "data" / "prepared" / "adult_income").exists():
            return candidate
    raise FileNotFoundError("Could not find data/prepared/adult_income from the current working directory.")


PROJECT_ROOT = resolve_project_root()
PREPARED_DIR = PROJECT_ROOT / "data" / "prepared" / "adult_income"
TRAIN_PREPARED_PATH = PREPARED_DIR / "train_prepared.csv"
TEST_PREPARED_PATH = PREPARED_DIR / "test_prepared.csv"
SAMPLE_SUBMISSION_PATH = PREPARED_DIR / "sample_submission.csv"
SUBMISSIONS_DIR = PROJECT_ROOT / "submissions"
SUBMISSIONS_DIR.mkdir(parents=True, exist_ok=True)

train_prepared = pd.read_csv(TRAIN_PREPARED_PATH)
test_prepared = pd.read_csv(TEST_PREPARED_PATH)
sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)

ID_COL = "Id"
TARGET_COL = "income"
NEGATIVE_LABEL = "<=50K"
POSITIVE_LABEL = ">50K"

assert ID_COL in train_prepared.columns, "train_prepared.csv must contain Id."
assert TARGET_COL in train_prepared.columns, "train_prepared.csv must contain income."
assert ID_COL in test_prepared.columns, "test_prepared.csv must contain Id."
assert TARGET_COL not in test_prepared.columns, "test_prepared.csv must not contain income."
assert sample_submission.columns.tolist() == [ID_COL, TARGET_COL], "sample_submission.csv must have columns Id and income."
assert len(sample_submission) == len(test_prepared), "sample_submission.csv row count must match test_prepared.csv."
assert set(train_prepared[TARGET_COL].unique()) == {NEGATIVE_LABEL, POSITIVE_LABEL}, "Unexpected training labels found in income."

print(f"Project root: {PROJECT_ROOT}")
print(f"Train prepared shape: {train_prepared.shape}")
print(f"Test prepared shape: {test_prepared.shape}")
print(f"Sample submission shape: {sample_submission.shape}")


## Compact kNN Feature Set

kNN is distance-sensitive, so this notebook intentionally uses a compact feature subset instead of every prepared feature from notebook 02. The goal is to keep the representation expressive without overloading the distance metric with redundant bins, duplicated missingness signals, or wide categorical interactions.


In [ ]:
NUMERIC_FEATURES = [
    "age",
    "education.num",
    "hours.per.week",
    "log1p_fnlwgt",
    "log1p_capital_gain",
    "log1p_capital_loss",
]

CATEGORICAL_FEATURES = [
    "workclass",
    "marital.status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "native.country",
]

SELECTED_FEATURE_COLUMNS = NUMERIC_FEATURES + CATEGORICAL_FEATURES

EXCLUDED_FEATURE_COLUMNS = [
    ID_COL,
    TARGET_COL,
    "fnlwgt",
    "capital.gain",
    "capital.loss",
    "age_band",
    "hours_ge_50",
    "hours_ge_60",
    "hours_ge_80",
    "capital_gain_positive",
    "capital_loss_positive",
    "workclass_missing",
    "occupation_missing",
    "native_country_missing",
    "education_level_group",
    "family_role_combo",
]

missing_train_features = sorted(set(SELECTED_FEATURE_COLUMNS) - set(train_prepared.columns))
missing_test_features = sorted(set(SELECTED_FEATURE_COLUMNS) - set(test_prepared.columns))
assert not missing_train_features, f"Missing selected features in train_prepared.csv: {missing_train_features}"
assert not missing_test_features, f"Missing selected features in test_prepared.csv: {missing_test_features}"

X_train_full = train_prepared[SELECTED_FEATURE_COLUMNS].copy()
X_test_final = test_prepared[SELECTED_FEATURE_COLUMNS].copy()
y_binary = (train_prepared[TARGET_COL] == POSITIVE_LABEL).astype(int)

feature_summary_df = pd.DataFrame(
    {
        "feature": SELECTED_FEATURE_COLUMNS,
        "feature_type": ["numeric"] * len(NUMERIC_FEATURES) + ["categorical"] * len(CATEGORICAL_FEATURES),
    }
)

excluded_feature_summary_df = pd.DataFrame({"excluded_feature": EXCLUDED_FEATURE_COLUMNS})

target_distribution_df = pd.DataFrame(
    {
        "income": train_prepared[TARGET_COL].value_counts().index,
        "count": train_prepared[TARGET_COL].value_counts().values,
        "share": train_prepared[TARGET_COL].value_counts(normalize=True).round(4).values,
    }
)

display(feature_summary_df)
display(excluded_feature_summary_df)
display(target_distribution_df)


In [ ]:
X_dev, X_holdout, y_dev, y_holdout = train_test_split(
    X_train_full,
    y_binary,
    test_size=0.20,
    random_state=42,
    stratify=y_binary,
)

split_summary_df = pd.DataFrame(
    [
        {
            "split": "development",
            "rows": len(X_dev),
            "positive_rate": round(float(y_dev.mean()), 4),
        },
        {
            "split": "holdout",
            "rows": len(X_holdout),
            "positive_rate": round(float(y_holdout.mean()), 4),
        },
    ]
)

display(split_summary_df)


## Preprocessing Pipeline and Hyperparameter Search

All preprocessing is fit inside the sklearn pipeline so the holdout set and competition test set remain untouched during model selection.


In [ ]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", RobustScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="__MISSING__")),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="infrequent_if_exist",
                min_frequency=20,
                sparse_output=False,
            ),
        ),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_transformer, NUMERIC_FEATURES),
        ("categorical", categorical_transformer, CATEGORICAL_FEATURES),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

base_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", KNeighborsClassifier(algorithm="brute", n_jobs=-1)),
    ]
)

cv_splitter = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

param_grid = {
    "model__n_neighbors": [3, 5, 7, 9, 11, 15, 21, 31, 41, 61],
    "model__weights": ["uniform", "distance"],
    "model__p": [1, 2],
}

grid_search = GridSearchCV(
    estimator=base_pipeline,
    param_grid=param_grid,
    scoring="accuracy",
    cv=cv_splitter,
    refit=True,
    n_jobs=-1,
    return_train_score=True,
)

grid_search.fit(X_dev, y_dev)

cv_results_df = pd.DataFrame(grid_search.cv_results_)[
    [
        "param_model__n_neighbors",
        "param_model__weights",
        "param_model__p",
        "mean_test_score",
        "std_test_score",
        "mean_train_score",
        "rank_test_score",
    ]
].rename(
    columns={
        "param_model__n_neighbors": "n_neighbors",
        "param_model__weights": "weights",
        "param_model__p": "p",
        "mean_test_score": "mean_cv_accuracy",
        "std_test_score": "std_cv_accuracy",
        "mean_train_score": "mean_train_accuracy",
        "rank_test_score": "grid_rank",
    }
)

cv_results_df["n_neighbors"] = cv_results_df["n_neighbors"].astype(int)
cv_results_df["p"] = cv_results_df["p"].astype(int)
cv_results_df["mean_cv_accuracy_rounded"] = cv_results_df["mean_cv_accuracy"].round(5)
cv_results_df["p_priority"] = cv_results_df["p"].map({1: 0, 2: 1})

cv_results_ranked_df = cv_results_df.sort_values(
    by=["mean_cv_accuracy_rounded", "n_neighbors", "p_priority"],
    ascending=[False, False, True],
).reset_index(drop=True)

selected_row = cv_results_ranked_df.iloc[0]
selected_params = {
    "model__n_neighbors": int(selected_row["n_neighbors"]),
    "model__weights": str(selected_row["weights"]),
    "model__p": int(selected_row["p"]),
}

selected_pipeline = clone(base_pipeline).set_params(**selected_params)
selected_pipeline.fit(X_dev, y_dev)

selection_summary_df = pd.DataFrame(
    [
        {
            "selection_source": "GridSearchCV raw best params",
            "params": json.dumps(grid_search.best_params_, sort_keys=True),
            "mean_cv_accuracy": round(float(grid_search.best_score_), 5),
        },
        {
            "selection_source": "Notebook tie-break selection",
            "params": json.dumps(selected_params, sort_keys=True),
            "mean_cv_accuracy": round(float(selected_row["mean_cv_accuracy"]), 5),
        },
    ]
)

display(cv_results_ranked_df.head(15))
display(selection_summary_df)
display(Markdown(f"**Best mean CV accuracy:** `{float(selected_row['mean_cv_accuracy']):.5f}`"))


## Holdout Evaluation

The selected model is evaluated exactly once on the untouched holdout split.


In [ ]:
holdout_pred = selected_pipeline.predict(X_holdout)
holdout_proba = selected_pipeline.predict_proba(X_holdout)[:, 1]

holdout_metrics = {
    "accuracy": float(accuracy_score(y_holdout, holdout_pred)),
    "balanced_accuracy": float(balanced_accuracy_score(y_holdout, holdout_pred)),
    "precision_positive": float(precision_score(y_holdout, holdout_pred, zero_division=0)),
    "recall_positive": float(recall_score(y_holdout, holdout_pred, zero_division=0)),
    "f1_positive": float(f1_score(y_holdout, holdout_pred, zero_division=0)),
    "roc_auc": float(roc_auc_score(y_holdout, holdout_proba)),
}

holdout_metrics_df = pd.DataFrame(
    [
        {"metric": metric_name, "value": round(metric_value, 5)}
        for metric_name, metric_value in holdout_metrics.items()
    ]
)

confusion_df = pd.DataFrame(
    confusion_matrix(y_holdout, holdout_pred),
    index=[f"actual_{NEGATIVE_LABEL}", f"actual_{POSITIVE_LABEL}"],
    columns=[f"pred_{NEGATIVE_LABEL}", f"pred_{POSITIVE_LABEL}"],
)

classification_report_df = pd.DataFrame(
    classification_report(
        y_holdout,
        holdout_pred,
        target_names=[NEGATIVE_LABEL, POSITIVE_LABEL],
        output_dict=True,
        zero_division=0,
    )
).transpose()

display(holdout_metrics_df)
display(confusion_df)
display(classification_report_df)


## Refit on Full Training Data and Generate Submission

After holdout evaluation, the selected pipeline is refit on the full prepared training set and used to score the competition test set.


In [ ]:
final_pipeline = clone(base_pipeline).set_params(**selected_params)
final_pipeline.fit(X_train_full, y_binary)

test_pred_binary = final_pipeline.predict(X_test_final)
test_pred_labels = np.where(test_pred_binary == 1, POSITIVE_LABEL, NEGATIVE_LABEL)

submission_df = pd.DataFrame(
    {
        ID_COL: test_prepared[ID_COL].values,
        TARGET_COL: test_pred_labels,
    }
)

assert submission_df.columns.tolist() == [ID_COL, TARGET_COL], "Submission columns must be Id and income."
assert len(submission_df) == len(test_prepared), "Submission row count must match test_prepared.csv."
assert set(submission_df[TARGET_COL].unique()).issubset({NEGATIVE_LABEL, POSITIVE_LABEL}), "Submission labels must be <=50K or >50K."

submission_path = SUBMISSIONS_DIR / "knn_submission.csv"
metrics_path = SUBMISSIONS_DIR / "knn_holdout_metrics.json"
best_params_path = SUBMISSIONS_DIR / "knn_best_params.json"

holdout_metrics_payload = {
    "selected_params": selected_params,
    "best_mean_cv_accuracy": float(selected_row["mean_cv_accuracy"]),
    "holdout_metrics": holdout_metrics,
    "holdout_confusion_matrix": confusion_df.to_dict(),
}

best_params_payload = {
    "grid_search_best_params": grid_search.best_params_,
    "tie_break_selected_params": selected_params,
    "tie_break_changed_selection": bool(grid_search.best_params_ != selected_params),
    "selected_mean_cv_accuracy": float(selected_row["mean_cv_accuracy"]),
    "selected_mean_cv_accuracy_rounded": float(selected_row["mean_cv_accuracy_rounded"]),
}

submission_df.to_csv(submission_path, index=False)
metrics_path.write_text(json.dumps(holdout_metrics_payload, indent=2))
best_params_path.write_text(json.dumps(best_params_payload, indent=2))

saved_artifacts_df = pd.DataFrame(
    [
        {"artifact": "submission", "path": str(submission_path.relative_to(PROJECT_ROOT))},
        {"artifact": "holdout_metrics", "path": str(metrics_path.relative_to(PROJECT_ROOT))},
        {"artifact": "best_params", "path": str(best_params_path.relative_to(PROJECT_ROOT))},
    ]
)

display(saved_artifacts_df)
display(submission_df.head())
display(Markdown(f"Submission saved to `{submission_path.relative_to(PROJECT_ROOT)}`."))
